## export files

In [1]:
# Export
#data_to_export = 'output/annual_arousal_indices.csv'
#data_to_export = 'output/annual_valence_indices.csv'
#data_to_export = 'output/breadth_scores.csv'
#data_to_export = 'output/ADHD_top_cooccurrences_by_decade_nostopwords.csv'

#data_to_export = 'output/annual_arousal_indices.csv'
#data_to_export = 'output/annual_arousal_indices_full_sentence.csv'
#data_to_export = 'output/annual_valence_indices.csv'
#data_to_export = 'output/annual_valence_indices_full_sentence.csv'
#data_to_export = 'output/breadth_scores.csv'

#data_to_export = 'output/annual_arousal_indices_nrc.csv'
#data_to_export = 'output/annual_valence_indices_nrc.csv'
#data_to_export = 'output/annual_ABSA_sentiment_indices.csv'
data_to_export = '2_step2_compute_breadth-score.py'

#data_to_export = 'output/valence_arousal_contributions/cooccurrences_by_year_valence_arousal_summary.txt'

#data_to_export = '1_step3_compute_ABSA_sentiment_by_year_sentences.py'

# Running the following cell will export the file from above by generating an email with a link to download the exported file. This link will expire after 2 hours.
# 30MB weekly limit to exporting
!aws s3 cp $data_to_export s3://pq-tdm-studio-results/tdm-ale-data/1656/results/

upload: ./2_step2_compute_breadth-score.py to s3://pq-tdm-studio-results/tdm-ale-data/1656/results/2_step2_compute_breadth-score.py


In [2]:
from pathlib import Path
import zipfile

# ============================================================
# SETTINGS
# ============================================================

folder = Path.cwd()

# Save outside project folder
zip_path = folder.parent / f"{folder.name}_PORTABLE_EXPORT.zip"

EXPORT_LIMIT_MB = 30
EXPORT_LIMIT_BYTES = EXPORT_LIMIT_MB * 1024 * 1024


# ============================================================
# FOLDERS TO EXCLUDE COMPLETELY
# ============================================================

excluded_folders = [
    Path("output/embeddings"),
    Path("0_corpus_processing/output/duplicatefiltering"),
]


# ============================================================
# SPECIFIC HUGE FILES TO EXCLUDE
# ============================================================

excluded_files = {
    # Very large processed sentence files
    Path("output/ADHD_US_newsstream_normalized-ADHD_sentences_cleaned_lemmatized.csv"),
    Path("output/ADHD_US_newsstream_normalized-ADHD_sentences.csv"),
    Path("output/ADHD_US_newsstream_normalized-ADHD_sentences_cleaned.csv"),

    # Large corpus-processing output
    Path("0_corpus_processing/output/ADHD_News_fulltext.csv"),
    Path("0_corpus_processing/output/ADHD_US_newsstream.csv"),

    # Large raw input files
    Path("input/ADHD_US_newsstream.tsv"),
    Path("input/ADHD_US_newsstream_normalized-ADHD.tsv"),
}


# ============================================================
# HELPER
# ============================================================

def should_exclude(path):
    rel = path.relative_to(folder)

    # Exact excluded files
    if rel in excluded_files:
        return True

    # Anything within excluded folders
    for excluded_folder in excluded_folders:
        try:
            rel.relative_to(excluded_folder)
            return True
        except ValueError:
            pass

    return False


# ============================================================
# COLLECT FILES
# ============================================================

all_files = [
    p for p in folder.rglob("*")
    if p.is_file()
]

included = []
excluded = []

for p in all_files:
    if should_exclude(p):
        excluded.append(p)
    else:
        included.append(p)


def total_mb(files):
    return sum(p.stat().st_size for p in files) / (1024 ** 2)


print(f"Total project files: {len(all_files):,}")
print(f"Included:            {len(included):,}")
print(f"Excluded:            {len(excluded):,}")

print(f"\nOriginal size: {total_mb(all_files):,.2f} MB")
print(f"Included size: {total_mb(included):,.2f} MB")
print(f"Excluded size: {total_mb(excluded):,.2f} MB")


# ============================================================
# CREATE ZIP
# ============================================================

print("\nCreating portable export...")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9
) as zf:

    for file_path in included:
        archive_name = file_path.relative_to(folder.parent)
        zf.write(file_path, archive_name)


# ============================================================
# CHECK FINAL COMPRESSED SIZE
# ============================================================

zip_mb = zip_path.stat().st_size / (1024 ** 2)

print("\n------------------------------------------")
print(f"ZIP created: {zip_path}")
print(f"FINAL ZIP SIZE: {zip_mb:.2f} MB")
print(f"EXPORT LIMIT:   {EXPORT_LIMIT_MB} MB")
print("------------------------------------------")


if zip_mb <= EXPORT_LIMIT_MB:
    print("\n✅ SUCCESS — archive is below the 30 MB export limit.")

else:
    print("\n❌ Still above the 30 MB export limit.")

    print("\nLargest files STILL INCLUDED:\n")

    largest = sorted(
        included,
        key=lambda p: p.stat().st_size,
        reverse=True
    )[:30]

    for p in largest:
        size = p.stat().st_size / (1024 ** 2)
        print(f"{size:9.2f} MB   {p.relative_to(folder)}")

Total project files: 627
Included:            303
Excluded:            324

Original size: 7,646.10 MB
Included size: 378.94 MB
Excluded size: 7,267.17 MB

Creating portable export...

------------------------------------------
ZIP created: /home/ec2-user/SageMaker/ADHD_conceptual_change_PORTABLE_EXPORT.zip
FINAL ZIP SIZE: 100.78 MB
EXPORT LIMIT:   30 MB
------------------------------------------

❌ Still above the 30 MB export limit.

Largest files STILL INCLUDED:

    74.22 MB   output/context_window_validation/adhd/all_context_window_arousal_scores.csv
    73.97 MB   output/context_window_validation/adhd/all_context_window_valence_scores.csv
    67.96 MB   output/context_window_validation/all_context_window_arousal_scores.csv
    67.90 MB   output/context_window_validation/all_context_window_valence_scores.csv
     5.79 MB   input/ADHD_US_newsstream_SAMPLE_1000.tsv
     3.93 MB   output/valence_arousal_contributions/ADHD_phase_valence_arousal_contributions.csv
     3.15 MB   0_corpu